# LeetCode 1533 — Find the Index of the Large Integer bằng Python

Link đề: https://leetcode.com/problems/find-the-index-of-the-large-integer/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Interactive

> **Ghi chú Premium:** đây là bài Premium (🔒). Đề bài, chữ ký và giới hạn số lần gọi trong notebook được viết theo hiểu biết về bài, đối chiếu với trang doocs (https://leetcode.doocs.org/en/lc/1533/). Nếu có tài khoản Premium, nên đối chiếu lại với đề gốc trên LeetCode. Lớp `ArrayReader` dùng trong notebook là **bản giả lập** tự viết để chạy thử.

Skeleton LeetCode (theo doocs):

```python
# """
# This is ArrayReader's API interface.
# You should not implement it, or speculate about its implementation
# """
# class ArrayReader(object):
#    # Compares the sum of arr[l..r] with the sum of arr[x..y]
#    # return 1 if sum(arr[l..r]) > sum(arr[x..y])
#    # return 0 if sum(arr[l..r]) == sum(arr[x..y])
#    # return -1 if sum(arr[l..r]) < sum(arr[x..y])
#    def compareSub(self, l: int, r: int, x: int, y: int) -> int:
#
#    # Returns the length of the array
#    def length(self) -> int:

class Solution:
    def getIndex(self, reader: 'ArrayReader') -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu dạng bài **tương tác (interactive)**: không truy cập trực tiếp mảng, chỉ được hỏi qua API với **giới hạn số lần gọi** (ở đây `compareSub` tối đa **20** lần).
2. Tự dựng **bộ giả lập** `ArrayReader` (giữ mảng ẩn, đếm số lần gọi, báo lỗi khi vượt giới hạn hoặc gọi sai tham số) để test lời giải.
3. Thấy vì sao so tuyến tính từng phần tử dùng quá nhiều lần gọi, và chuyển sang **chia đôi** (cân hai nửa bằng nhau) với `⌈log₂ n⌉ ≤ 19` lần gọi.
4. Cải tiến thành **chia ba** (lời giải doocs): mỗi lần gọi có 3 kết quả nên loại được 2/3 → `⌈log₃ n⌉ ≤ 12` lần gọi, và đây là **tối ưu** theo lập luận thông tin.
5. Rèn kỹ năng xử lý chỉ số biên: hai đoạn đem so phải **dài bằng nhau**, phần dư phải được tính đúng.

## 2. Đề bài & ví dụ

Có mảng số nguyên `arr` trong đó **mọi phần tử bằng nhau, trừ đúng một phần tử lớn hơn** các phần tử còn lại. Ta không được đọc `arr` trực tiếp mà chỉ dùng đối tượng `reader` (kiểu `ArrayReader`) với hai hàm:

- `compareSub(l, r, x, y)` (với `0 ≤ l, r, x, y < length()`, `l ≤ r`, `x ≤ y`): so sánh tổng `arr[l..r]` với tổng `arr[x..y]` (đoạn **bao gồm hai đầu**), trả về `1` nếu tổng thứ nhất lớn hơn, `0` nếu bằng, `-1` nếu nhỏ hơn.
- `length()`: trả về độ dài mảng.

Được gọi `compareSub` **tối đa 20 lần**; coi cả hai hàm chạy `O(1)`. Trả về **chỉ số** của phần tử lớn nhất.

**Ví dụ 1:** `arr = [7,7,7,7,10,7,7,7]` → `4`. Một chuỗi gọi hợp lệ: `compareSub(0,0,1,1)` → `0` (loại 0, 1), `compareSub(2,2,3,3)` → `0` (loại 2, 3), `compareSub(4,4,5,5)` → `1` ⇒ chỉ số 4. Dùng 3 lần gọi.

**Ví dụ 2:** `nums = [6,6,12]` → `2`.

**Ràng buộc:** `2 ≤ arr.length ≤ 5·10⁵`, `1 ≤ arr[i] ≤ 100`, mọi phần tử bằng nhau trừ một phần tử lớn hơn tất cả.

**Follow-up (từ đề):** nếu có **hai** số lớn hơn tất cả các số khác thì sao? Nếu có một số lớn hơn và một số nhỏ hơn các số khác thì sao? (bàn ở mục 15)

### Bộ giả lập `ArrayReader` (bản giả lập tự viết — không phải code của LeetCode)

- Giữ mảng ẩn `_arr` và mảng tổng tiền tố để `compareSub` chạy `O(1)`.
- Đếm `calls`; khi vượt `limit` (mặc định 20) thì đánh dấu `exceeded = True` và ném `CallLimitExceeded`.
- Kiểm tra tham số đúng ràng buộc (`l ≤ r`, `x ≤ y`, trong phạm vi), sai thì ném `ValueError`.
- Hàm `run(solution_cls, arr)` chạy lời giải, **bắt** lỗi và trả về bộ `(kết quả, số lần gọi, trạng thái)` — notebook không bao giờ bị văng exception.

In [1]:
import math
import random
import time
from typing import List


class CallLimitExceeded(Exception):
    pass


class ArrayReader:
    """BẢN GIẢ LẬP ArrayReader của LeetCode 1533 (tự viết để test)."""

    def __init__(self, arr: List[int], limit: int = 20, log: bool = False):
        self._arr = list(arr)
        self._pre = [0]
        for v in self._arr:
            self._pre.append(self._pre[-1] + v)
        self.limit = limit
        self.calls = 0
        self.exceeded = False
        self.log = log

    def _sum(self, a, b):
        return self._pre[b + 1] - self._pre[a]

    def compareSub(self, l: int, r: int, x: int, y: int) -> int:
        n = len(self._arr)
        if not (0 <= l <= r < n and 0 <= x <= y < n):
            raise ValueError(f"tham số sai: compareSub({l}, {r}, {x}, {y}) với n={n}")
        self.calls += 1
        if self.calls > self.limit:
            self.exceeded = True
            raise CallLimitExceeded(f"vượt quá {self.limit} lần gọi compareSub")
        s1, s2 = self._sum(l, r), self._sum(x, y)
        res = (s1 > s2) - (s1 < s2)
        if self.log:
            print(f"  compareSub({l}, {r}, {x}, {y}) -> {res}")
        return res

    def length(self) -> int:
        return len(self._arr)


def large_index(arr):
    """Đáp án thật (chỉ người kiểm tra mới biết): chỉ số phần tử lớn nhất."""
    return max(range(len(arr)), key=arr.__getitem__)


def run(solution_cls, arr, limit=20, log=False):
    reader = ArrayReader(arr, limit, log)
    try:
        got = solution_cls().getIndex(reader)
    except CallLimitExceeded as e:
        return None, reader.calls, "VƯỢT GIỚI HẠN: " + str(e)
    except ValueError as e:
        return None, reader.calls, "GỌI SAI: " + str(e)
    status = "ĐÚNG" if got == large_index(arr) else f"SAI (đáp án {large_index(arr)})"
    return got, reader.calls, status


examples = [([7, 7, 7, 7, 10, 7, 7, 7], 4), ([6, 6, 12], 2)]
rd = ArrayReader(examples[0][0], log=True)
print("Tái hiện chuỗi gọi của ví dụ 1:")
print(rd.compareSub(0, 0, 1, 1), rd.compareSub(2, 2, 3, 3), rd.compareSub(4, 4, 5, 5), "| số lần gọi:", rd.calls)

Tái hiện chuỗi gọi của ví dụ 1:
  compareSub(0, 0, 1, 1) -> 0
  compareSub(2, 2, 3, 3) -> 0
  compareSub(4, 4, 5, 5) -> 1
0 0 1 | số lần gọi: 3


## 3. Kiến thức nền

### 3.1 Bài tương tác (interactive) là gì?

Thay vì nhận dữ liệu, lời giải nhận một **đối tượng** và phải "hỏi" nó. Điểm số không chỉ phụ thuộc kết quả đúng mà còn **số câu hỏi**. Vì vậy ta phân tích độ phức tạp theo **số lần gọi API** là chính, thời gian chạy là phụ.

### 3.2 Tổng tiền tố (prefix sum)

`pre[i]` = tổng `i` phần tử đầu. Tổng đoạn `arr[a..b] = pre[b+1] − pre[a]` trong `O(1)` — bộ giả lập dùng kỹ thuật này.

In [2]:
arr = [7, 7, 10, 7]
pre = [0]
for v in arr:
    pre.append(pre[-1] + v)
print("pre =", pre)
print("tổng arr[1..2] =", pre[3] - pre[1], "| kiểm tra:", sum(arr[1:3]))

pre = [0, 7, 14, 24, 31]
tổng arr[1..2] = 17 | kiểm tra: 17


### 3.3 So sánh ba chiều không dùng `if`

`(a > b) - (a < b)` cho `1`, `0` hoặc `-1` (vì `True = 1`, `False = 0`) — giống hàm `cmp` cũ của Python 2.

In [3]:
for a, b in [(5, 3), (4, 4), (2, 9)]:
    print(a, b, "->", (a > b) - (a < b))

5 3 -> 1
4 4 -> 0
2 9 -> -1


### 3.4 Logarit và số bước chia nhỏ

Nếu mỗi bước giữ lại tối đa `⌈m / k⌉` phần tử từ `m`, thì số bước để còn 1 phần tử là `⌈log_k n⌉`. Tính bằng số nguyên để tránh sai số dấu phẩy động: lặp `m = ceil(m / k)` tới khi `m == 1`.

In [4]:
def steps(n, k):
    s = 0
    while n > 1:
        n = -(-n // k)          # ceil(n / k) bằng số nguyên
        s += 1
    return s

for n in [2, 3, 8, 9, 10, 1000, 5 * 10**5]:
    print(f"n={n:>6}: chia đôi {steps(n, 2):>2} bước | chia ba {steps(n, 3):>2} bước")

n=     2: chia đôi  1 bước | chia ba  1 bước
n=     3: chia đôi  2 bước | chia ba  1 bước
n=     8: chia đôi  3 bước | chia ba  2 bước
n=     9: chia đôi  4 bước | chia ba  2 bước
n=    10: chia đôi  4 bước | chia ba  3 bước
n=  1000: chia đôi 10 bước | chia ba  7 bước
n=500000: chia đôi 19 bước | chia ba 12 bước


### 3.5 Exception tự định nghĩa và `try/except`

Ta tạo lớp `CallLimitExceeded(Exception)` để phân biệt lỗi "vượt giới hạn" với lỗi khác, và bắt nó trong `run` để báo cáo gọn gàng.

In [5]:
try:
    r = ArrayReader([1, 2], limit=1)
    r.compareSub(0, 0, 1, 1)
    r.compareSub(0, 0, 1, 1)          # lần thứ 2 > limit
except CallLimitExceeded as e:
    print("Bắt được:", e, "| exceeded =", r.exceeded)

Bắt được: vượt quá 1 lần gọi compareSub | exceeded = True


## 4. Ý tưởng nền tảng / trực giác

Bài này chính là câu đố **"tìm đồng xu nặng bằng cân đĩa"**: `compareSub` là một chiếc cân hai đĩa. Nếu đặt lên hai đĩa **cùng số lượng** phần tử:

- Cân bằng (`0`) ⇒ phần tử lớn **không** nằm trên đĩa nào — nó ở phần còn lại.
- Lệch ⇒ phần tử lớn nằm trên đĩa **nặng hơn**.

Làm tay với `arr = [7,7,7,7,10,7,7,7]` (n = 8):

1. Chia đôi: đĩa trái `[0..3]` (tổng 28), đĩa phải `[4..7]` (tổng 31) → `-1` ⇒ phần tử lớn trong `[4..7]`.
2. `[4..5]` vs `[6..7]`: 17 vs 14 → `1` ⇒ trong `[4..5]`.
3. `[4]` vs `[5]`: 10 vs 7 → `1` ⇒ chỉ số 4. Ba lần cân.

Chú ý: nếu hai đĩa **không cùng số lượng**, kết quả vô nghĩa (đĩa nhiều phần tử hơn có thể nặng hơn dù không chứa phần tử lớn).

In [6]:
arr = [7, 7, 7, 7, 10, 7, 7, 7]
rd = ArrayReader(arr, log=True)
print(rd.compareSub(0, 3, 4, 7))
print(rd.compareSub(4, 5, 6, 7))
print(rd.compareSub(4, 4, 5, 5))
print("=> chỉ số 4 sau", rd.calls, "lần gọi")

  compareSub(0, 3, 4, 7) -> -1
-1
  compareSub(4, 5, 6, 7) -> 1
1
  compareSub(4, 4, 5, 5) -> 1
1
=> chỉ số 4 sau 3 lần gọi


## 5. Cách 1 — Brute Force / baseline: so tuyến tính từng cặp

Làm như chuỗi gọi trong ví dụ 1: so `arr[0]` với `arr[1]`, `arr[2]` với `arr[3]`, … Gặp cặp lệch thì trả về phần tử nặng hơn. Nếu `n` lẻ và mọi cặp đều cân bằng thì phần tử cuối là đáp án.

In [7]:
class SolutionLinear:
    def getIndex(self, reader: 'ArrayReader') -> int:
        n = reader.length()
        for i in range(0, n - 1, 2):
            c = reader.compareSub(i, i, i + 1, i + 1)
            if c == 1:
                return i
            if c == -1:
                return i + 1
        return n - 1                  # n lẻ, mọi cặp bằng nhau

for arr, want in examples:
    print(arr, "->", run(SolutionLinear, arr))

[7, 7, 7, 7, 10, 7, 7, 7] -> (4, 3, 'ĐÚNG')
[6, 6, 12] -> (2, 1, 'ĐÚNG')


Với mảng nhỏ thì đúng, nhưng số lần gọi worst case là `⌊n/2⌋`. Với `n` chỉ cần ≥ 42 và phần tử lớn nằm cuối là đã vượt 20 lần:

In [8]:
for n in [10, 41, 42, 1000, 5 * 10**5]:
    arr = [5] * n
    arr[-1] = 9
    print(f"n={n:>6}:", run(SolutionLinear, arr))

n=    10: (9, 5, 'ĐÚNG')
n=    41: (40, 20, 'ĐÚNG')
n=    42: (None, 21, 'VƯỢT GIỚI HẠN: vượt quá 20 lần gọi compareSub')
n=  1000: (None, 21, 'VƯỢT GIỚI HẠN: vượt quá 20 lần gọi compareSub')
n=500000: (None, 21, 'VƯỢT GIỚI HẠN: vượt quá 20 lần gọi compareSub')


**Độ phức tạp:** `O(n)` lần gọi (worst `⌊n/2⌋`), `O(1)` bộ nhớ. **Nút thắt:** mỗi lần cân chỉ loại được **2 phần tử**. Ta cần mỗi lần cân loại được **một tỷ lệ cố định** của khoảng đang xét.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đặt lên mỗi đĩa **cả một đoạn** cùng độ dài. Một lần cân loại được hàng nghìn phần tử cùng lúc.

**Invariant:** phần tử lớn luôn nằm trong đoạn `[left, right]`. Ban đầu `[0, n−1]`; dừng khi `left == right`.

**Quy tắc chia đôi** (đoạn có `m = right − left + 1` phần tử, `half = m // 2`):
- So `[left, left+half−1]` với `[left+half, left+2·half−1]` (hai đoạn **cùng dài** `half`).
- Nếu `1` thì giữ nửa trái; nếu `-1` thì giữ nửa phải.
- Nếu `0` thì phần tử lớn là phần tử **thừa** (chỉ xảy ra khi `m` lẻ): `left = right`.

**Quy tắc chia ba** (doocs): cắt thành hai đoạn đem cân **cùng dài** và một đoạn dư:
- Nếu `0` thì phần tử lớn ở đoạn dư; nếu `1` ở đoạn thứ nhất; nếu `-1` ở đoạn thứ hai.
- Ba kết quả ⇒ mỗi lần giữ lại ~1/3 ⇒ `⌈log₃ n⌉` lần gọi.

## 7. Các cách tiếp theo

### Cách 2 — Chia đôi (binary search bằng cân hai nửa)

Từng đoạn:
- `half = (right - left + 1) // 2`: số phần tử mỗi đĩa.
- Hai đĩa: `[left, left + half - 1]` và `[left + half, left + 2*half - 1]`.
- Phần tử thừa (nếu có) là `right` khi `m` lẻ (vì `left + 2*half − 1 = right − 1`).

In [9]:
def binary_demo(arr):
    reader = ArrayReader(arr, log=True)
    left, right = 0, reader.length() - 1
    while left < right:
        half = (right - left + 1) // 2
        a1, b1, a2, b2 = left, left + half - 1, left + half, left + 2 * half - 1
        c = reader.compareSub(a1, b1, a2, b2)
        if c == 1:
            right = b1
        elif c == -1:
            left, right = a2, b2
        else:
            left = right            # phần tử thừa duy nhất
        print(f"    -> khoảng còn lại [{left}, {right}]")
    print("kết quả:", left, "| số lần gọi:", reader.calls)
    return left

binary_demo([3, 3, 3, 3, 3, 3, 8])

  compareSub(0, 2, 3, 5) -> 0
    -> khoảng còn lại [6, 6]
kết quả: 6 | số lần gọi: 1


6

Code nộp LeetCode cho cách 2:

In [10]:
class SolutionBinary:
    def getIndex(self, reader: 'ArrayReader') -> int:
        left, right = 0, reader.length() - 1
        while left < right:
            half = (right - left + 1) // 2
            cmp = reader.compareSub(left, left + half - 1, left + half, left + 2 * half - 1)
            if cmp == 1:
                right = left + half - 1
            elif cmp == -1:
                left, right = left + half, left + 2 * half - 1
            else:
                left = right
        return left

for arr, want in examples:
    print(arr, "->", run(SolutionBinary, arr))

[7, 7, 7, 7, 10, 7, 7, 7] -> (4, 3, 'ĐÚNG')
[6, 6, 12] -> (2, 1, 'ĐÚNG')


**Số lần gọi:** mỗi bước giữ lại `half = ⌊m/2⌋` phần tử (hoặc chỉ 1 phần tử thừa khi cân bằng), nên worst case `⌊log₂ n⌋` lần — với `n = 5·10⁵` là **18** lần, nằm trong giới hạn 20. Time `O(log n)`, Space `O(1)`.

### Cách 3 — Chia ba (lời giải doocs, tối ưu số lần gọi)

Với `k = (right − left) // 3`:
- `t1 = left`, `t2 = left + k`: đoạn thứ nhất `[t1, t2]` có `k + 1` phần tử.
- `t3 = left + 2k + 1`: đoạn thứ hai `[t2 + 1, t3]` cũng có `k + 1` phần tử.
- Đoạn dư `[t3 + 1, right]` có `m − 2(k + 1)` phần tử (có thể rỗng — khi đó kết quả `0` không thể xảy ra vì phần tử lớn phải nằm trên một đĩa).

Kích thước lớn nhất trong ba phần là `⌈m / 3⌉` (kiểm tra ở cell dưới).

In [11]:
def parts(m):
    left, right = 0, m - 1
    k = (right - left) // 3
    t2, t3 = left + k, left + 2 * k + 1
    return (t2 - left + 1, t3 - t2, right - t3)

for m in range(2, 13):
    p = parts(m)
    print(f"m={m:>2}: kích thước (đĩa 1, đĩa 2, dư) = {p}, max = {max(p)}, ceil(m/3) = {-(-m // 3)}")
    assert p[0] == p[1] and sum(p) == m and max(p) == -(-m // 3)

m= 2: kích thước (đĩa 1, đĩa 2, dư) = (1, 1, 0), max = 1, ceil(m/3) = 1
m= 3: kích thước (đĩa 1, đĩa 2, dư) = (1, 1, 1), max = 1, ceil(m/3) = 1
m= 4: kích thước (đĩa 1, đĩa 2, dư) = (2, 2, 0), max = 2, ceil(m/3) = 2
m= 5: kích thước (đĩa 1, đĩa 2, dư) = (2, 2, 1), max = 2, ceil(m/3) = 2
m= 6: kích thước (đĩa 1, đĩa 2, dư) = (2, 2, 2), max = 2, ceil(m/3) = 2
m= 7: kích thước (đĩa 1, đĩa 2, dư) = (3, 3, 1), max = 3, ceil(m/3) = 3
m= 8: kích thước (đĩa 1, đĩa 2, dư) = (3, 3, 2), max = 3, ceil(m/3) = 3
m= 9: kích thước (đĩa 1, đĩa 2, dư) = (3, 3, 3), max = 3, ceil(m/3) = 3
m=10: kích thước (đĩa 1, đĩa 2, dư) = (4, 4, 2), max = 4, ceil(m/3) = 4
m=11: kích thước (đĩa 1, đĩa 2, dư) = (4, 4, 3), max = 4, ceil(m/3) = 4
m=12: kích thước (đĩa 1, đĩa 2, dư) = (4, 4, 4), max = 4, ceil(m/3) = 4


In [12]:
def ternary_demo(arr):
    reader = ArrayReader(arr, log=True)
    left, right = 0, reader.length() - 1
    while left < right:
        t1, t2, t3 = left, left + (right - left) // 3, left + ((right - left) // 3) * 2 + 1
        cmp = reader.compareSub(t1, t2, t2 + 1, t3)
        if cmp == 0:
            left = t3 + 1
        elif cmp == 1:
            right = t2
        else:
            left, right = t2 + 1, t3
        print(f"    -> khoảng còn lại [{left}, {right}]")
    print("kết quả:", left, "| số lần gọi:", reader.calls)
    return left

ternary_demo([7, 7, 7, 7, 10, 7, 7, 7])

  compareSub(0, 2, 3, 5) -> -1
    -> khoảng còn lại [3, 5]
  compareSub(3, 3, 4, 4) -> -1
    -> khoảng còn lại [4, 4]
kết quả: 4 | số lần gọi: 2


4

Code nộp LeetCode cho cách 3 (giống lời giải Python của doocs):

In [13]:
class SolutionTernary:
    def getIndex(self, reader: 'ArrayReader') -> int:
        left, right = 0, reader.length() - 1
        while left < right:
            t1, t2, t3 = (
                left,
                left + (right - left) // 3,
                left + ((right - left) // 3) * 2 + 1,
            )
            cmp = reader.compareSub(t1, t2, t2 + 1, t3)
            if cmp == 0:
                left = t3 + 1
            elif cmp == 1:
                right = t2
            else:
                left, right = t2 + 1, t3
        return left

for arr, want in examples:
    print(arr, "->", run(SolutionTernary, arr))

[7, 7, 7, 7, 10, 7, 7, 7] -> (4, 2, 'ĐÚNG')
[6, 6, 12] -> (2, 1, 'ĐÚNG')


**Số lần gọi:** `⌈log₃ n⌉` — với `n = 5·10⁵` là **12** lần. Time `O(log n)`, Space `O(1)`.

### Cách 4 — Chia đôi viết theo đệ quy (để hiểu "cân" như chia để trị)

Cùng ý tưởng cách 2 nhưng viết dạng hàm `solve(left, right)`; giúp người mới thấy cấu trúc chia để trị. Độ sâu đệ quy chỉ `O(log n)` nên an toàn.

In [14]:
class SolutionRecursive:
    def getIndex(self, reader: 'ArrayReader') -> int:
        def solve(left, right):
            if left == right:
                return left
            half = (right - left + 1) // 2
            cmp = reader.compareSub(left, left + half - 1, left + half, left + 2 * half - 1)
            if cmp == 1:
                return solve(left, left + half - 1)
            if cmp == -1:
                return solve(left + half, left + 2 * half - 1)
            return right                       # phần tử thừa
        return solve(0, reader.length() - 1)

for arr, want in examples:
    print(arr, "->", run(SolutionRecursive, arr))

[7, 7, 7, 7, 10, 7, 7, 7] -> (4, 3, 'ĐÚNG')
[6, 6, 12] -> (2, 1, 'ĐÚNG')


## 8. TRACE từng bước

Bảng trạng thái của cách 3 (chia ba) với `n = 20`, phần tử lớn ở chỉ số `13`:

| Bước | left | right | t2 | t3 | Đĩa 1 | Đĩa 2 | Dư | cmp | Khoảng mới |
|------|------|-------|----|----|-------|-------|-----|-----|------------|
| 1 | 0 | 19 | 6 | 13 | [0..6] | [7..13] | [14..19] | -1 | [7, 13] |
| 2 | 7 | 13 | 9 | 12 | [7..9] | [10..12] | [13..13] | 0 | [13, 13] |

Chỉ 2 lần gọi. Hàm `trace_ternary` in đầy đủ các biến qua từng vòng lặp.

In [15]:
def trace_ternary(arr):
    reader = ArrayReader(arr)
    left, right = 0, reader.length() - 1
    step = 0
    print(f"{'bước':>4} | {'left':>4} {'right':>5} | {'t2':>3} {'t3':>3} | đĩa 1     đĩa 2      dư         | cmp | khoảng mới")
    while left < right:
        step += 1
        t1, t2, t3 = left, left + (right - left) // 3, left + ((right - left) // 3) * 2 + 1
        cmp = reader.compareSub(t1, t2, t2 + 1, t3)
        old = (left, right)
        if cmp == 0:
            left = t3 + 1
        elif cmp == 1:
            right = t2
        else:
            left, right = t2 + 1, t3
        rest = f"[{t3 + 1}..{old[1]}]" if t3 + 1 <= old[1] else "rỗng"
        print(f"{step:>4} | {old[0]:>4} {old[1]:>5} | {t2:>3} {t3:>3} | " + f"[{t1}..{t2}]".ljust(10)
              + f"[{t2 + 1}..{t3}]".ljust(11) + rest.ljust(11) + f"| {cmp:>3} | [{left}, {right}]")
    print("kết quả:", left, "| số lần gọi:", reader.calls)
    return left

arr = [4] * 20
arr[13] = 5
trace_ternary(arr)

bước | left right |  t2  t3 | đĩa 1     đĩa 2      dư         | cmp | khoảng mới
   1 |    0    19 |   6  13 | [0..6]    [7..13]    [14..19]   |  -1 | [7, 13]
   2 |    7    13 |   9  12 | [7..9]    [10..12]   [13..13]   |   0 | [13, 13]
kết quả: 13 | số lần gọi: 2


13

Trace thêm cách 2 (chia đôi) trên cùng mảng để so sánh số lần gọi:

In [16]:
def trace_binary(arr):
    reader = ArrayReader(arr)
    left, right, step = 0, len(arr) - 1, 0
    while left < right:
        step += 1
        half = (right - left + 1) // 2
        cmp = reader.compareSub(left, left + half - 1, left + half, left + 2 * half - 1)
        msg = f"bước {step}: [{left}..{left + half - 1}] vs [{left + half}..{left + 2 * half - 1}] -> {cmp:>2}"
        if cmp == 1:
            right = left + half - 1
        elif cmp == -1:
            left, right = left + half, left + 2 * half - 1
        else:
            left = right
        print(msg, f"| khoảng mới [{left}, {right}]")
    print("kết quả:", left, "| số lần gọi:", reader.calls)

trace_binary(arr)

bước 1: [0..9] vs [10..19] -> -1 | khoảng mới [10, 19]
bước 2: [10..14] vs [15..19] ->  1 | khoảng mới [10, 14]
bước 3: [10..11] vs [12..13] -> -1 | khoảng mới [12, 13]
bước 4: [12..12] vs [13..13] -> -1 | khoảng mới [13, 13]
kết quả: 13 | số lần gọi: 4


## 9. Code nộp LeetCode

Bản nộp chính: chia ba (đúng tên `Solution.getIndex(self, reader: 'ArrayReader') -> int` như doocs). Trên LeetCode không được tự định nghĩa `ArrayReader`; lớp ở trên chỉ là giả lập.

In [17]:
class Solution:
    def getIndex(self, reader: 'ArrayReader') -> int:
        left, right = 0, reader.length() - 1
        while left < right:
            k = (right - left) // 3
            t2 = left + k                 # đĩa 1: [left, t2]       (k + 1 phần tử)
            t3 = left + 2 * k + 1         # đĩa 2: [t2 + 1, t3]     (k + 1 phần tử)
            cmp = reader.compareSub(left, t2, t2 + 1, t3)
            if cmp == 0:                  # cân bằng -> ở phần dư
                left = t3 + 1
            elif cmp == 1:                # đĩa 1 nặng hơn
                right = t2
            else:                         # đĩa 2 nặng hơn
                left, right = t2 + 1, t3
        return left

for arr, want in examples:
    print(arr, "->", run(Solution, arr))

[7, 7, 7, 7, 10, 7, 7, 7] -> (4, 2, 'ĐÚNG')
[6, 6, 12] -> (2, 1, 'ĐÚNG')


## 10. Vì sao thuật toán đúng

**Bổ đề cân:** gọi `v` là giá trị chung, `V > v` là giá trị lớn. Hai đoạn **cùng độ dài** `len`: tổng của đoạn không chứa phần tử lớn là `len·v`; của đoạn chứa nó là `len·v + (V − v) > len·v`. Vậy `compareSub` trả `1`/`-1` chỉ đúng phía chứa phần tử lớn, và `0` ⇔ không đĩa nào chứa nó.

**Invariant:** phần tử lớn luôn trong `[left, right]`. Ban đầu đúng. Mỗi vòng, ba phần (đĩa 1, đĩa 2, dư) **phủ kín** `[left, right]` và không giao nhau; theo bổ đề cân, ta giữ lại đúng phần chứa phần tử lớn ⇒ invariant được bảo toàn. Nếu phần dư rỗng thì `cmp = 0` không thể xảy ra (phần tử lớn phải nằm trên một đĩa), nên không bao giờ có `left > right`.

**Kết thúc:** mỗi vòng đoạn giảm còn tối đa `⌈m/3⌉ < m` (với `m ≥ 2`) ⇒ dừng; khi `left == right`, phần tử duy nhất còn lại là đáp án.

**Tối ưu:** mỗi lần gọi chỉ có 3 kết quả, nên `q` lần gọi phân biệt được tối đa `3^q` vị trí. Cần `3^q ≥ n` ⇒ `q ≥ ⌈log₃ n⌉` — chia ba đạt đúng cận dưới này.

## 11. Phân tích Time / Space

Gọi `n = reader.length()`.

| Cách | Số lần gọi `compareSub` (worst) | n = 5·10⁵ | Time | Space |
|------|-------------------------------|-----------|------|-------|
| 1. Tuyến tính theo cặp | `⌊n/2⌋` | 250 000 ✘ | `O(n)` | `O(1)` |
| 2. Chia đôi | `⌊log₂ n⌋` | 18 ✔ | `O(log n)` | `O(1)` |
| 3. Chia ba | `⌈log₃ n⌉` | 12 ✔ | `O(log n)` | `O(1)` |
| 4. Chia đôi đệ quy | `⌊log₂ n⌋` | 18 ✔ | `O(log n)` | `O(log n)` stack |

Bộ giả lập tốn thêm `O(n)` để dựng tổng tiền tố — đó là chi phí của "trọng tài", không tính vào lời giải. Cell dưới đo số lần gọi worst case thực tế bằng cách thử mọi vị trí với vài giá trị `n`.

In [18]:
def worst_calls(sol_cls, n):
    worst = 0
    base = [1] * n
    for pos in range(n):
        base[pos] = 2
        got, calls, status = run(sol_cls, base, limit=10**9)
        assert status == "ĐÚNG", (sol_cls.__name__, n, pos, status)
        worst = max(worst, calls)
        base[pos] = 1
    return worst

for n in [2, 3, 9, 10, 27, 28, 100, 243, 500]:
    print(f"n={n:>3}: chia đôi {worst_calls(SolutionBinary, n):>2} | chia ba {worst_calls(Solution, n):>2}"
          f" | log2 sàn {n.bit_length() - 1:>2} | ceil(log3) {steps(n, 3):>2}")

n=  2: chia đôi  1 | chia ba  1 | log2 sàn  1 | ceil(log3)  1
n=  3: chia đôi  1 | chia ba  1 | log2 sàn  1 | ceil(log3)  1
n=  9: chia đôi  3 | chia ba  2 | log2 sàn  3 | ceil(log3)  2
n= 10: chia đôi  3 | chia ba  3 | log2 sàn  3 | ceil(log3)  3
n= 27: chia đôi  4 | chia ba  3 | log2 sàn  4 | ceil(log3)  3
n= 28: chia đôi  4 | chia ba  4 | log2 sàn  4 | ceil(log3)  4
n=100: chia đôi  6 | chia ba  5 | log2 sàn  6 | ceil(log3)  5
n=243: chia đôi  7 | chia ba  5 | log2 sàn  7 | ceil(log3)  5
n=500: chia đôi  8 | chia ba  6 | log2 sàn  8 | ceil(log3)  6


## 12. Bộ test

- Ví dụ đề; edge case `n = 2` (cả hai vị trí), `n = 3`, phần tử lớn ở đầu/cuối/giữa.
- **Mọi vị trí** với mọi `n ≤ 120`: kiểm tra đúng và số lần gọi ≤ 20 (cho các cách 2, 3, 4).
- Mảng ngẫu nhiên seed cố định: `n` ngẫu nhiên tới `5·10⁵`, giá trị trong `[1, 100]`.
- Cách 1 phải **bị bắt** khi vượt giới hạn.

In [19]:
GOOD = [SolutionBinary, SolutionTernary, SolutionRecursive, Solution]
for arr, want in examples + [([2, 1], 0), ([1, 2], 1), ([5, 1, 1], 0), ([1, 5, 1], 1), ([100, 99], 0)]:
    for S in GOOD + [SolutionLinear]:
        got, calls, status = run(S, arr)
        assert got == want and status == "ĐÚNG", (S.__name__, arr, got, calls, status)
print("ví dụ + edge: OK")

total = 0
for n in range(2, 121):
    for pos in range(n):
        arr = [3] * n
        arr[pos] = 4
        for S in GOOD:
            got, calls, status = run(S, arr)
            assert status == "ĐÚNG" and calls <= 20, (S.__name__, n, pos, calls, status)
        total += 1
print(f"Mọi vị trí với n = 2..120 ({total} mảng): OK, luôn ≤ 20 lần gọi")

ví dụ + edge: OK


Mọi vị trí với n = 2..120 (7259 mảng): OK, luôn ≤ 20 lần gọi


In [20]:
rng = random.Random(1533)
max_calls = {S.__name__: 0 for S in GOOD}
t0 = time.perf_counter()
for it in range(40):
    n = rng.choice([rng.randint(2, 50), rng.randint(2, 5000), rng.randint(400000, 500000)])
    v = rng.randint(1, 99)
    big = rng.randint(v + 1, 100)
    arr = [v] * n
    arr[rng.randrange(n)] = big
    for S in GOOD:
        got, calls, status = run(S, arr)
        assert status == "ĐÚNG", (S.__name__, n, status)
        max_calls[S.__name__] = max(max_calls[S.__name__], calls)
print("40 mảng ngẫu nhiên (seed 1533): OK | số lần gọi lớn nhất:", max_calls, f"| {time.perf_counter() - t0:.2f}s")

40 mảng ngẫu nhiên (seed 1533): OK | số lần gọi lớn nhất: {'SolutionBinary': 18, 'SolutionTernary': 12, 'SolutionRecursive': 18, 'Solution': 12} | 2.11s


In [21]:
arr = [7] * 500000
arr[-1] = 8
print("Cách 1 trên n = 5·10^5:", run(SolutionLinear, arr))
arr2 = [7] * 500000
arr2[0] = 8
print("Cách 1, phần tử lớn ở đầu (may mắn):", run(SolutionLinear, arr2))

Cách 1 trên n = 5·10^5: (None, 21, 'VƯỢT GIỚI HẠN: vượt quá 20 lần gọi compareSub')
Cách 1, phần tử lớn ở đầu (may mắn): (0, 1, 'ĐÚNG')


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chia đôi kiểu `mid` thông thường, hai đĩa không cùng độ dài

`mid = (left + right) // 2`, so `[left, mid]` với `[mid + 1, right]`. Khi số phần tử lẻ, đĩa trái nhiều hơn một phần tử ⇒ đĩa trái có thể nặng hơn dù không chứa phần tử lớn.

In [22]:
class WrongMid:
    def getIndex(self, reader):
        left, right = 0, reader.length() - 1
        while left < right:
            mid = (left + right) // 2
            cmp = reader.compareSub(left, mid, mid + 1, right)   # SAI khi độ dài lẻ
            if cmp >= 0:
                right = mid
            else:
                left = mid + 1
        return left

for arr in [[6, 6, 12], [7, 7, 7, 7, 10, 7, 7, 7], [5, 5, 5, 5, 6]]:
    print(arr, "-> sai:", run(WrongMid, arr))

[6, 6, 12] -> sai: (0, 2, 'SAI (đáp án 2)')
[7, 7, 7, 7, 10, 7, 7, 7] -> sai: (4, 3, 'ĐÚNG')
[5, 5, 5, 5, 6] -> sai: (0, 3, 'SAI (đáp án 4)')


Sửa: hai đĩa phải có **cùng** `half = m // 2` phần tử, phần tử thừa xét riêng (`SolutionBinary`).

### Lỗi 2 — Tính `t3` sai trong chia ba (thiếu `+ 1`)

Viết `t3 = left + 2k` làm đĩa 2 có `k` phần tử trong khi đĩa 1 có `k + 1` ⇒ so sánh lệch, giống lỗi 1.

In [23]:
class WrongT3:
    def getIndex(self, reader):
        left, right = 0, reader.length() - 1
        guard = 0
        while left < right and guard < 100:
            guard += 1
            k = (right - left) // 3
            t2, t3 = left + k, left + 2 * k           # SAI: phải + 1
            if t3 < t2 + 1:                             # đĩa 2 rỗng -> không gọi được
                return -1
            cmp = reader.compareSub(left, t2, t2 + 1, t3)
            if cmp == 0:
                left = t3 + 1
            elif cmp == 1:
                right = t2
            else:
                left, right = t2 + 1, t3
        return left

rng_e = random.Random(2)
for _ in range(4):
    n = rng_e.randint(5, 30)
    arr = [5] * n
    arr[rng_e.randrange(n)] = 6
    print(f"n={n:>2}, đúng={large_index(arr):>2} -> sai:", run(WrongT3, arr))

n= 6, đúng= 0 -> sai: (-1, 1, 'SAI (đáp án 0)')
n= 7, đúng= 2 -> sai: (-1, 1, 'SAI (đáp án 2)')
n=10, đúng= 4 -> sai: (-1, 2, 'SAI (đáp án 4)')
n=13, đúng= 9 -> sai: (-1, 2, 'SAI (đáp án 9)')


Sửa: `t3 = left + 2k + 1` để hai đĩa đều có `k + 1` phần tử (đã kiểm chứng bằng hàm `parts` ở mục 7).

### Lỗi 3 — Nhầm chiều kết quả `1` / `-1`

`compareSub(l, r, x, y) = 1` nghĩa là **đoạn thứ nhất** nặng hơn. Đảo ngược hai nhánh sẽ luôn đi về phía nhẹ.

In [24]:
class WrongDirection:
    def getIndex(self, reader):
        left, right = 0, reader.length() - 1
        while left < right:
            k = (right - left) // 3
            t2, t3 = left + k, left + 2 * k + 1
            cmp = reader.compareSub(left, t2, t2 + 1, t3)
            if cmp == 0:
                left = t3 + 1
            elif cmp == -1:                 # SAI: đảo nhánh
                right = t2
            else:
                left, right = t2 + 1, t3
        return left

for arr, want in examples + [([1, 1, 1, 1, 1, 2, 1, 1, 1], 5)]:
    print(arr, "-> sai:", run(WrongDirection, arr), "| đúng:", want)

[7, 7, 7, 7, 10, 7, 7, 7] -> sai: (2, 2, 'SAI (đáp án 4)') | đúng: 4
[6, 6, 12] -> sai: (2, 1, 'ĐÚNG') | đúng: 2
[1, 1, 1, 1, 1, 2, 1, 1, 1] -> sai: (2, 2, 'SAI (đáp án 5)') | đúng: 5


Sửa: `cmp == 1` ⇒ giữ đĩa 1 (`right = t2`); `cmp == -1` ⇒ giữ đĩa 2.

### Lỗi 4 — So tuyến tính / so từng phần tử với `arr[0]`

Đúng về kết quả nhưng vượt giới hạn 20 lần gọi; bộ giả lập báo ngay.

In [25]:
class WrongScanFirst:
    def getIndex(self, reader):
        n = reader.length()
        for i in range(1, n):
            c = reader.compareSub(0, 0, i, i)
            if c == 1:
                return 0
            if c == -1:
                return i
        return 0

arr = [2] * 100
arr[77] = 3
print("sai:", run(WrongScanFirst, arr), "| đúng:", run(Solution, arr))

sai: (None, 21, 'VƯỢT GIỚI HẠN: vượt quá 20 lần gọi compareSub') | đúng: (77, 4, 'ĐÚNG')


Sửa: dùng chia đôi/chia ba để số lần gọi là `O(log n)`.

### Lỗi 5 — Truyền đoạn rỗng hoặc ngược (`l > r`)

Khi `m = 2`, công thức "dư" có thể rỗng; nếu ai đó cố so đoạn dư với một đĩa thì gọi `compareSub(l, r, …)` với `l > r` — vi phạm ràng buộc. Bộ giả lập phát hiện và báo "GỌI SAI".

In [26]:
class WrongEmpty:
    def getIndex(self, reader):
        left, right = 0, reader.length() - 1
        while left < right:
            k = (right - left) // 3
            t2, t3 = left + k, left + 2 * k + 1
            cmp = reader.compareSub(t3 + 1, right, left, t2)   # SAI: đem đoạn dư (có thể rỗng) đi cân
            if cmp == 1:
                left = t3 + 1
            else:
                right = t3
        return left

print([run(WrongEmpty, a) for a in ([6, 6, 12], [9, 1])])

[(2, 1, 'ĐÚNG'), (None, 0, 'GỌI SAI: tham số sai: compareSub(2, 1, 0, 0) với n=2')]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mọi phần tử giống nhau trừ một"** + chỉ được so sánh tổng ⇒ câu đố **cân đồng xu**: chia thành các nhóm **cùng kích thước**.
- Giới hạn số câu hỏi kiểu `20` với `n ≤ 5·10⁵` ⇒ cần `O(log n)` câu hỏi; `2²⁰ ≈ 10⁶ > 5·10⁵` gợi ý chia đôi là đủ, chia ba còn dư nhiều.
- Mỗi truy vấn có `k` kết quả ⇒ cận dưới `⌈log_k n⌉`; thiết kế để mỗi kết quả loại được phần bằng nhau.
- Bài tương tác: **luôn viết bộ giả lập** có đếm lần gọi và kiểm tra tham số trước khi nộp.
- Họ bài tương tác cùng kiểu: *Guess Number Higher or Lower* (374), *First Bad Version* (278), *Search in a Sorted Array of Unknown Size* (702), *Guess the Majority in a Hidden Array* (1538), *Find in Mountain Array* (1095).

## 15. Biến thể của bài

1. **Phần tử lệch nhỏ hơn** (một phần tử nhẹ): giữ nguyên thuật toán, chỉ đảo vai trò `1`/`-1`.
2. **Không biết lệch nặng hay nhẹ** (bài 12 đồng xu cổ điển): cần thêm câu hỏi để xác định chiều; với `q` lần cân phân biệt được `(3^q − 3)/2` đồng xu.
3. **Follow-up 1 — hai số lớn hơn mọi số khác** (giả sử hai số lớn bằng nhau): khi chia đôi, nếu hai đĩa cân bằng thì có thể mỗi đĩa chứa một số lớn — cần chiến lược khác (so một đĩa với đoạn đã biết là "nhẹ"), số câu hỏi khoảng `2·log n`.
4. **Follow-up 2 — một số lớn và một số nhỏ:** cân bằng không còn nghĩa "không chứa" (số lớn và số nhỏ có thể bù nhau nếu độ lệch bằng nhau), nên phải tìm trước một đoạn tham chiếu "bình thường".
5. **Giới hạn số lần gọi chặt hơn**, ví dụ chỉ `⌈log₃ n⌉`: chỉ chia ba mới qua được — cell dưới chứng minh chia đôi thất bại khi `limit = 12`, `n = 5·10⁵`.

In [27]:
rng_v = random.Random(15)
arr = [9] * 500000
fails = 0
for _ in range(5):
    pos = rng_v.randrange(len(arr))
    arr[pos] = 10
    b = run(SolutionBinary, arr, limit=12)
    t = run(Solution, arr, limit=12)
    print(f"pos={pos:>6}: chia đôi -> {b[2]:45s} | chia ba -> {t[2]} ({t[1]} lần)")
    arr[pos] = 9

pos=109565: chia đôi -> VƯỢT GIỚI HẠN: vượt quá 12 lần gọi compareSub | chia ba -> ĐÚNG (12 lần)


pos=  6110: chia đôi -> VƯỢT GIỚI HẠN: vượt quá 12 lần gọi compareSub | chia ba -> ĐÚNG (12 lần)


pos=273320: chia đôi -> VƯỢT GIỚI HẠN: vượt quá 12 lần gọi compareSub | chia ba -> ĐÚNG (12 lần)
pos=385871: chia đôi -> VƯỢT GIỚI HẠN: vượt quá 12 lần gọi compareSub | chia ba -> ĐÚNG (12 lần)


pos= 18966: chia đôi -> VƯỢT GIỚI HẠN: vượt quá 12 lần gọi compareSub | chia ba -> ĐÚNG (12 lần)


## 16. So sánh các cách

| Cách | Ý tưởng | Lần gọi | Time | Space | Khi nào dùng |
|------|---------|---------|------|-------|--------------|
| 1. Tuyến tính theo cặp | cân 2 phần tử một lần | `⌊n/2⌋` | `O(n)` | `O(1)` | chỉ để hiểu đề; vượt giới hạn |
| 2. Chia đôi | cân hai nửa bằng nhau, xét phần tử thừa | `⌊log₂ n⌋` | `O(log n)` | `O(1)` | dễ viết, đủ qua giới hạn 20 |
| 3. Chia ba (doocs) | hai đĩa + phần dư, 3 kết quả | `⌈log₃ n⌉` | `O(log n)` | `O(1)` | **tối ưu**, bản nộp chính |
| 4. Chia đôi đệ quy | như 2, dạng chia để trị | `⌊log₂ n⌋` | `O(log n)` | `O(log n)` | minh hoạ tư duy đệ quy |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Chạy `Solution` (chia ba) trên `[7,7,7,7,10,7,7,7]`: lần gọi **đầu tiên** là `compareSub(?, ?, ?, ?)` và trả về gì? Tổng cộng mấy lần gọi?

**Bài 2 (tính tay):** Với `n = 5·10⁵`, số lần gọi worst case của chia đôi và chia ba là bao nhiêu? Còn `n = 3¹⁰ = 59049`?

**Bài 3 (điền TODO):** Hoàn thiện lớp `Ex3Binary` (chia đôi) trong cell dưới.

**Bài 4 (tự viết lại):** Biến thể **phần tử nhỏ**: mọi phần tử bằng nhau trừ một phần tử **nhỏ hơn**. Viết `Ex4Small.getIndex(reader)` dùng chia ba.

**Bài 5 (mini challenge):** Viết `max_len(q)` = độ dài lớn nhất của mảng mà luôn tìm được phần tử lớn với **tối đa `q`** lần gọi. Kiểm chứng bằng bộ giả lập: chia ba thành công với `limit = q` cho **mọi vị trí** khi `n = max_len(q)`, và với `n = max_len(q) + 1` thì có vị trí khiến nó cần `q + 1` lần.

In [28]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
class Ex3Binary:
    def getIndex(self, reader: 'ArrayReader') -> int:
        left, right = 0, reader.length() - 1
        while left < right:
            half = (right - left + 1) // 2
            # TODO: gọi compareSub với hai đĩa cùng dài half
            # TODO: cập nhật left/right theo 3 trường hợp 1, -1, 0
            pass
        return left

In [29]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
class Ex4Small:
    def getIndex(self, reader: 'ArrayReader') -> int:
        # TODO
        raise NotImplementedError

def max_len(q: int) -> int:
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `left = 0, right = 7`, `k = 7 // 3`. (2) `t2 = 2`, `t3 = 5` ⇒ cân `[0..2]` với `[3..5]`. (3) 21 vs 24 → `-1`; khoảng mới `[3, 5]`, rồi cân `[3]` vs `[4]` → `-1` ⇒ 2 lần gọi.

**Bài 2.** (1) Chia đôi: `⌊log₂ n⌋`; `2¹⁸ = 262144 ≤ 5·10⁵ < 2¹⁹`. (2) Chia ba: `⌈log₃ n⌉`; `3¹¹ = 177147 < 5·10⁵ ≤ 3¹² = 531441`. (3) `n = 3¹⁰`: chia ba đúng 10; chia đôi `⌊log₂ 59049⌋ = 15`.

**Bài 3.** (1) Hai đĩa `[left, left+half−1]`, `[left+half, left+2·half−1]`. (2) `1` → `right = left+half−1`; `-1` → `left, right = left+half, left+2·half−1`. (3) `0` → `left = right`.

**Bài 4.** (1) Đĩa chứa phần tử nhỏ sẽ **nhẹ** hơn. (2) Đổi nhánh: `cmp == -1` ⇒ giữ đĩa 1. (3) `cmp == 0` vẫn là phần dư.

**Bài 5.** (1) Mỗi lần gọi có 3 kết quả. (2) `q` lần gọi phân biệt tối đa `3^q` khả năng ⇒ `max_len(q) = 3^q`. (3) Dùng `worst_calls`-style vòng lặp qua mọi vị trí với `limit = q`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
rd = ArrayReader([7, 7, 7, 7, 10, 7, 7, 7], log=True)
a1 = Solution().getIndex(rd)
print("Bài 1: kết quả", a1, "| số lần gọi", rd.calls)
assert a1 == 4 and rd.calls == 2

# Bài 2
a2 = (steps(5 * 10**5, 2), steps(5 * 10**5, 3), (5 * 10**5).bit_length() - 1, steps(3**10, 3), (3**10).bit_length() - 1)
print("Bài 2: ceil(log2)=%d, ceil(log3)=%d, chia đôi worst=%d | n=3^10: chia ba %d, chia đôi %d" % a2)
assert a2 == (19, 12, 18, 10, 15)
assert worst_calls(SolutionBinary, 3**5) == (3**5).bit_length() - 1 and worst_calls(Solution, 3**5) == 5

  compareSub(0, 2, 3, 5) -> -1
  compareSub(3, 3, 4, 4) -> -1
Bài 1: kết quả 4 | số lần gọi 2
Bài 2: ceil(log2)=19, ceil(log3)=12, chia đôi worst=18 | n=3^10: chia ba 10, chia đôi 15


In [31]:
# Bài 3
class Ex3BinarySol:
    def getIndex(self, reader):
        left, right = 0, reader.length() - 1
        while left < right:
            half = (right - left + 1) // 2
            cmp = reader.compareSub(left, left + half - 1, left + half, left + 2 * half - 1)
            if cmp == 1:
                right = left + half - 1
            elif cmp == -1:
                left, right = left + half, left + 2 * half - 1
            else:
                left = right
        return left

for n in range(2, 80):
    for pos in range(n):
        arr = [1] * n
        arr[pos] = 2
        assert run(Ex3BinarySol, arr)[2] == "ĐÚNG"
print("Bài 3: OK")

Bài 3: OK


In [32]:
# Bài 4
class Ex4SmallSol:
    def getIndex(self, reader):
        left, right = 0, reader.length() - 1
        while left < right:
            k = (right - left) // 3
            t2, t3 = left + k, left + 2 * k + 1
            cmp = reader.compareSub(left, t2, t2 + 1, t3)
            if cmp == 0:
                left = t3 + 1
            elif cmp == -1:          # đĩa 1 nhẹ hơn -> chứa phần tử nhỏ
                right = t2
            else:
                left, right = t2 + 1, t3
        return left

def small_index(arr):
    return min(range(len(arr)), key=arr.__getitem__)

rng4 = random.Random(4)
for _ in range(300):
    n = rng4.randint(2, 300)
    v = rng4.randint(2, 100)
    arr = [v] * n
    p = rng4.randrange(n)
    arr[p] = rng4.randint(1, v - 1)
    rd = ArrayReader(arr)
    assert Ex4SmallSol().getIndex(rd) == p == small_index(arr) and rd.calls <= 20
print("Bài 4: OK")

Bài 4: OK


In [33]:
# Bài 5
def max_len_sol(q):
    return 3 ** q

def worst_with_limit(n, limit):
    ok_all = True
    for pos in range(n):
        arr = [1] * n
        arr[pos] = 2
        if run(Solution, arr, limit=limit)[2] != "ĐÚNG":
            ok_all = False
    return ok_all

for q in range(1, 6):
    n = max_len_sol(q)
    assert worst_with_limit(n, q), q                 # 3^q phần tử: luôn đủ q lần
    assert not worst_with_limit(n + 1, q), q         # 3^q + 1: có vị trí cần q + 1 lần
print("Bài 5: OK — max_len(q) = 3^q, ví dụ max_len(12) =", max_len_sol(12))

Bài 5: OK — max_len(q) = 3^q, ví dụ max_len(12) = 531441


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def getIndex(self, reader: 'ArrayReader') -> int:
        left, right = 0, reader.length() - 1
        while left < right:
            k = (right - left) // 3
            t2, t3 = left + k, left + 2 * k + 1      # [left,t2] và [t2+1,t3] cùng k+1 phần tử
            cmp = reader.compareSub(left, t2, t2 + 1, t3)
            if cmp == 0:   left = t3 + 1             # ở phần dư
            elif cmp == 1: right = t2                # đĩa 1 nặng
            else:          left, right = t2 + 1, t3  # đĩa 2 nặng
        return left
```

**Template "cân đồng xu / tìm phần tử lệch bằng truy vấn so sánh":** giữ invariant "đáp án trong `[left, right]`"; mỗi truy vấn chia thành các nhóm **cùng kích thước** + phần dư; mỗi kết quả loại các nhóm không chứa đáp án.

**Checklist trước khi submit:**
- [ ] Hai đoạn đem so **cùng độ dài** (kiểm tra với `m` chẵn, lẻ, `m = 2`, `m = 3`).
- [ ] Không bao giờ gọi `compareSub` với đoạn rỗng (`l > r`).
- [ ] `1` = đoạn **thứ nhất** nặng hơn.
- [ ] Số lần gọi worst case ≤ 20 với `n = 5·10⁵` (chia đôi 18, chia ba 12).
- [ ] Đúng chữ ký `getIndex(self, reader: 'ArrayReader') -> int`; không tự định nghĩa `ArrayReader` khi nộp.